In [1]:
import os
from pathlib import Path

import polars as pl

In [2]:
DATABASE_URI = os.environ["DATABASE_URI"]

# Annotation manuelle (Christian)

In [3]:
DATASETS_PATH = Path(
    "/Users/luis/projets/beta.gouv/qfdmod/quefairedemesobjets/data-platform/notebooks/deduplication/datasets"
)

In [4]:
def create_entity_pairs_from_datasets(datasets_folder_path: Path) -> pl.DataFrame:
    csv_files_to_load = datasets_folder_path.glob("*.csv")

    dfs = []
    for csv_filepath in csv_files_to_load:
        df = (
            pl.read_csv(csv_filepath).lazy().rename({"Good ? ": "Good ?"}, strict=False)
        )
        df_pairs = (
            df.join(df, on="cluster_id")
            .filter(pl.col("Good ?").is_in(["oui", "non"]))
            .with_columns(
                pl.min_horizontal(
                    pl.col("identifiant_unique"), pl.col("identifiant_unique_right")
                ).alias("identifiant_unique_i"),
                pl.max_horizontal(
                    pl.col("identifiant_unique"), pl.col("identifiant_unique_right")
                ).alias("identifiant_unique_j"),
                (pl.col("Good ?") == "oui").alias("label"),
            )
            .filter(
                (pl.col("identifiant_unique_i") != pl.col("identifiant_unique_j"))
                & (
                    pl.col("cluster_id")
                    .cum_count()
                    .over(
                        ["cluster_id", "identifiant_unique_i", "identifiant_unique_j"]
                    )
                    == 1
                )
            )
            .select(["identifiant_unique_i", "identifiant_unique_j", "label"])
        )
        dfs.append(df_pairs)

    df_pairs_concat = (
        pl.concat(dfs)
        .unique(["identifiant_unique_i", "identifiant_unique_j"])
        .sort(["identifiant_unique_i", "identifiant_unique_j"])
        .collect()
    )

    return df_pairs_concat

In [5]:
df_manual_labeling = create_entity_pairs_from_datasets(DATASETS_PATH)

In [6]:
df_manual_labeling

identifiant_unique_i,identifiant_unique_j,label
str,str,bool
"""07e07779-388e-5971-947b-b999ae…","""lokki_67692fc641d8badbf051ec32""",true
"""085352c0-77c6-599d-b54a-02ef2a…","""772950c8-5202-5394-b0ae-8ebbda…",false
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5487990001""",false
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488009001""",false
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488015001""",false
…,…,…
"""reseau_vrac_reemploi_14u_47.55…","""reseau_vrac_reemploi_1EK_47_-2""",true
"""reseau_vrac_reemploi_1DB_45_4""","""reseau_vrac_reemploi_tR_45.118…",true
"""reseau_vrac_reemploi_wA_44.818…","""reseau_vrac_reemploi_wC_44.900…",false


# Depuis la base

## Examples négatifs

In [7]:
sql_method_1 = """
with suggestions_avec_creation_de_parent as -- Sélection des suggestions de clustering validées dont l'objet était de lier un acteur avec un parent (existant ou à créer)
(
select
	ds.id,
	ds.statut,
	jsonb_path_query_array(ds.contexte, '$.fuzzy_details[*].identifiant_unique') as identifiant_unique_list,
	ds.suggestion->'changes'->0->'model_params'->>'id' as id_parent,
	ds.suggestion->'changes'->0->>'reason' as suggestion_reason
from
	data_suggestion ds
inner join data_suggestioncohorte ds2 
on
	ds.suggestion_cohorte_id = ds2.id
	and ds2.type_action = 'CLUSTERING'
where
	ds.contexte->'fuzzy_details' is not null
	and ds.suggestion->'changes'->0->>'reason' in ('1️⃣ 1 seul parent existant → à garder','➕ Pas de parent → à créer')
	and ds.statut='SUCCES'
),
suggestions_explosees as ( -- un couple (acteur, parent) par ligne
select
	s.id,
	s.id_parent,
	s.suggestion_reason,
	s_lid.id_acteur
from
	suggestions_avec_creation_de_parent s,
	 jsonb_array_elements_text(s.identifiant_unique_list) as s_lid(id_acteur)
where id_acteur<>s.id_parent
),
parents as (
select distinct parent_id from qfdmo_revisionacteur where parent_id is not null
)
select
	 qv.identifiant_unique as acteur_identitifiant_unique,
	 se.id_parent as suggestion_parent_id,
	 se.id as suggestion_id,
	 se.suggestion_reason
from
	qfdmo_revisionacteur qv
inner join suggestions_explosees se on qv.identifiant_unique = se.id_acteur
where
	qv.identifiant_unique not in (select parent_id from parents) -- acteur n'est pas un parent
	and qv.parent_id is null -- Acteur n'a pas de parent
	and qv.statut <> 'SUPPRIME' -- Acteur n'est pas supprimé
"""

In [9]:
df_negatives_from_db = pl.read_database_uri(query=sql_method_1,uri=DATABASE_URI)

In [10]:
df_negatives_from_db

acteur_identitifiant_unique,suggestion_parent_id,suggestion_id,suggestion_reason
str,str,i32,str
"""ecomaison_2094982""","""7a50dbdf-6775-54b3-8fda-f1a361…",670059,"""1️⃣ 1 seul parent existant → à…"
"""ocab_ecominero_35140835600019-…","""a4a7885b-a77d-50e1-98a0-41fcf5…",691084,"""1️⃣ 1 seul parent existant → à…"
"""ocab_valobat_79097077600023-li…","""5fc53c3e-d111-405c-aaa7-04519c…",693020,"""1️⃣ 1 seul parent existant → à…"
"""ocab_ecominero_53743318700466-…","""378d39d8-dff9-5fb6-bba8-15b8d1…",673618,"""1️⃣ 1 seul parent existant → à…"
"""valdelia_169263""","""b7882f2f-1c1f-4e17-8596-952837…",669887,"""1️⃣ 1 seul parent existant → à…"
…,…,…,…
"""ocab_ecominero_63672012000212-…","""0c0da61e-3451-5406-a2f8-f6523a…",691485,"""1️⃣ 1 seul parent existant → à…"
"""ocab_ecominero_75034416000064-…","""3b45afd0-b629-494c-9cee-cd83ec…",691552,"""1️⃣ 1 seul parent existant → à…"
"""refashion_TLC-REFASHION-PAV-34…","""c6da2cf0-ba12-45be-aa2f-f4c340…",653179,"""1️⃣ 1 seul parent existant → à…"
